# Homework 07

In this exercise, you will practice inferential statistics with confidence intervals, bootstrapping, and hypothesis testing. Problems may involve a combination of math and code. 

Recall that you can use LaTeX to nicely format your math inside Markdown cellsby enclosing equations in single dollar signs (e.g., $x^2+4=8$) for inline math or double dollar signs for centered equations like $$P(X > 5) = \frac{1}{6}.$$ See Worked Examples/HW 05 for further instructions and more examples.

Show your work and/or briefly explain your answers. In general, you will not receive full credit for numeric answers with no accompanying work or justification (math, code, explanation). For numeric answers, we will accept answers that are very slightly off due to rounding, $z$-score of $2$ vs. $1.96$, etc. 

### Scipy Version Check
The autograder uses `scipy` version 1.9.0+, if you do not have this version or higher, you will fail autograder tests. This cell is to ensure you have a high enough version.

In [1]:
try:
    import scipy
    print(scipy.__version__)
    if (scipy.__version__ < '1.10.0'): 
        %pip install --upgrade scipy
except Exception as e:
    %pip install scipy

1.17.1


In [2]:
# Run this code cell to import relevant libraries
import numpy as np
import pandas as pd
from scipy import stats

### Question 1 (4 points, all autograded)

A website is trying to increase registration for first-time visitors, exposing a random subset of these visitors to a new site design. Of $752$ randomly sampled visitors over a month who saw the new design, $64$ registered. Using [`stats.norm.interval`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html), construct a $95\%$ confidence interval for the percentage of visitors who would register for the website under the new design. 

Save your answer in a tuple `q1_1` of two `float` or `numpy.float64` variables such that `q1_1[0]` is the lower/left bound and `q1_1[1]` is the upper/right bound of your confidence interval. We accept either percentages (e.g., $50.0$) or fractions (e.g., $0.5$). [`stats.norm.interval`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html) when used correctly already returns a tuple.

In [3]:
# Code for question 1

percent_register = 64 / 752
se = np.sqrt(percent_register * (1 - percent_register) / 752)

q1 = stats.norm.interval(0.95, loc = percent_register, scale = se)

# Leave these lines here for grading and ease of debugging
print("Q1: 95% confidence interval: ", q1)

Q1: 95% confidence interval:  (np.float64(0.06516269200219607), np.float64(0.10505007395525073))


### Question 2 (8 points: 4 autograded, 4 manually graded)

A study examined the mean pay for a random sample of men and women entering the workforce as doctors for $43$ different positions. If each gender was equally paid, then we would expect about half of those positions to have men paid more than women and women would be paid more than men in the other half of positions. In the study, men were, on average, paid more in $29$ of the $43$ positions. 

Complete a hypothesis test using `stats.norm.cdf` to examine whether there is significant evidence (at the $0.05$ level) of gender discrimination in pay in these positions. It can be either one-sided or two-sided. Uncomment the corresponding line to indicate which test you are conducting in the variable `q2_setting`. Save your p-value in `q2` as `float` or `numpy.float64`. 

In the written part:
  1. Explicitly write down the null hypothesis and the alternative hypothesis for this test.
  1. Report your p-value and interpret the result.
  1. Explain why using `stats.norm.cdf` (not `stats.t.cdf`) is a good choice for this context.

In [4]:
# Code for question 2
q2_setting = 1 # Uncomment for one-sided test
# q2_setting = 2 # Uncomment for two-sided test

percent_gender = 29 / 43
p0 = 0.5

se = np.sqrt(p0 * (1 - p0) / 43)

z = (percent_gender - p0) / se

q2 = 1 - stats.norm.cdf(z)

# Leave these lines here for grading and ease of debugging
print("Q2: p-value = {} for {}-sided test".format(q2, q2_setting))

Q2: p-value = 0.01108395287649444 for 1-sided test


<!-- BEGIN QUESTION -->

### Answer 2

1. Let $p$ be the true proportion of positions in which men are paid more than women.

\begin{align*}
H_0 &: p = 0.5 \\
H_A &: p > 0.5
\end{align*}

The null hypothesis states that men and women are equally likely to be paid more across positions.  
The alternative hypothesis states that men are paid more than women in more than half of positions.

2. The sample proportion is:
\begin{align*}
\hat{p} &= \frac{29}{43} \approx 0.674.
\end{align*}

The standard error is:
\begin{align*}
SE &= \sqrt{\frac{p_0(1-p_0)}{n}} \\
   &= \sqrt{\frac{0.5(1-0.5)}{43}} \\
   &\approx 0.076.
\end{align*}

The test statistic is:
\begin{align*}
z &= \frac{\hat{p} - p_0}{SE} \\
  &= \frac{0.674 - 0.5}{0.076} \\
  &\approx 2.28.
\end{align*}

The p-value is:
\begin{align*}
\text{p-value} &= P(Z > 2.28) \\
               &= 1 - \Phi(2.28) \\
               &\approx 0.011.
\end{align*}

Since the p-value ($\approx 0.011$) is less than $0.05$, we reject the null hypothesis.  
Thus there is statistically significant evidence that men are paid more than women in more than half of these positions, suggesting possible gender discrimination in pay.

3. This is a hypothesis test for a population proportion. The sampling distribution of $\hat{p}$ is approximately normal when:

\begin{align*}
np_0 &\ge 10 \\
n(1 - p_0) &\ge 10.
\end{align*}

Here,
\begin{align*}
43(0.5) &= 21.5 \ge 10.
\end{align*}

Thus, the normal approximation is valid, and using `stats.norm.cdf` is appropriate.  
A t-distribution is used for inference about population means when the population standard deviation is unknown, which is not appplicable here.

<!-- END QUESTION -->

## Movie Ratings Data
In the remainder of this assignment you will work with the movielens dataset of movie ratings that we have seen before. Below we import and preview the data. It consists of 2 tables: `users` has a row for every individual who has rated any movies, `movie-ratings` has a row for every rating of a particular movie by a particular user. This means users with multiple ratings are in the `movie_ratings` multiple times. The data is a random sample of all of the movie ratings made on the movielens service.

In [5]:
users = pd.read_csv("users.csv")
users.head()

,user_id,age,sex,occupation
0,1,24,M,technician
1,2,53,F,other
2,3,23,M,writer
3,4,24,M,technician
4,5,33,F,other


In [6]:
movie_ratings = pd.read_csv("movies-all.csv")
movie_ratings.head()

,user_id,age,sex,occupation,movie_id,rating,movie_title
0,1,24,M,technician,61,4,Three Colors: White (1994)
1,13,47,M,educator,61,4,Three Colors: White (1994)
2,18,35,F,other,61,4,Three Colors: White (1994)
3,58,27,M,programmer,61,5,Three Colors: White (1994)
4,59,49,M,educator,61,4,Three Colors: White (1994)


### Question 3 (12 points: 8 autograded, 4 manually graded)

1. Compute a $95\%$ confidence interval for the mean `age` of users using `stats.norm.interval`. Save your answer in a tuple `q3_1` similar to that in Q1.
2. Compute a $95\%$ confidence interval for the mean `age` of users **who have rated the movie `Casablanca (1942)`** using the normal distribution. Save your answer in a tuple `q3_2`.

In the written part, answer the following question.

3. `Casablanca (1942)` is an old movie, one might suspect that it has been rated by older individuals on average than the entire dataset. Just looking at the confidence intervals you computed in steps 1 and 2, can you conclude that there is significant evidence for this belief? Why or why not? 

In [7]:
# Code for question 3
mean_all = np.mean(users["age"])
se_all = np.std(users["age"], ddof=1) / np.sqrt(len(users["age"]))

q3_1 = stats.norm.interval(0.95, loc=mean_all, scale=se_all)

casablanca_ages = movie_ratings[movie_ratings["movie_title"] == "Casablanca (1942)"]["age"].dropna()

mean_casa = np.mean(casablanca_ages)
se_casa = np.std(casablanca_ages, ddof=1) / np.sqrt(len(casablanca_ages))

q3_2 = stats.norm.interval(0.95, loc=mean_casa, scale=se_casa)

# Leave these lines here for grading and ease of debugging
print("3.1: 95% confidence interval: ", q3_1)
print("3.2: 95% confidence interval: ", q3_2)


# Leave these lines here for grading and ease of debugging
print("3.1: 95% confidence interval: ", q3_1)
print("3.2: 95% confidence interval: ", q3_2)

3.1: 95% confidence interval:  (np.float64(33.27375766393051), np.float64(34.830165984001624))
3.2: 95% confidence interval:  (np.float64(34.460497318851566), np.float64(37.33374136427601))
3.1: 95% confidence interval:  (np.float64(33.27375766393051), np.float64(34.830165984001624))
3.2: 95% confidence interval:  (np.float64(34.460497318851566), np.float64(37.33374136427601))


<!-- BEGIN QUESTION -->

## Answer 3

Looking at the two 95\% confidence intervals alone, we cannot automatically conclude that users who rated `Casablanca (1942)` are older on average.

If the two confidence intervals overlap, this suggests that the difference in average age may not be statistically significant at the 5\% level. In other words, the true mean ages could still be similar.

Even if the intervals do not overlap, simply comparing two separate confidence intervals is not the same as formally testing whether the two means are different. To properly determine whether users who rated `Casablanca (1942)` are significantly older, we would need to perform a two-sample hypothesis test or construct a confidence interval for the difference in means.

Therefore, based only on the two confidence intervals, we cannot definitively conclude that `Casablanca (1942)` was rated by older individuals on average.

<!-- END QUESTION -->

### Question 4 (6 points, all autograded)
Only $18$ users have rated the movie `Lost in Space (1998)`.
1. Use bootstrapping with $10000$ bootstrap resamples to compute a $95\%$ confidence interval for the mean `age` of users who have rated `Lost in Space (1998)`. Save your answer in a tuple `q4_1`.
2. One of the advantages of bootstrapping is that we can easily compute confidence intervals for arbitrary measurements of distributions. Use bootstrapping with $10000$ bootstrap resamples to compute a $95\%$ confidence interval for the **median** `rating` of `Lost in Space (1998)`. Note that `numpy` provides a vectorized function for [calculating the median](https://numpy.org/doc/stable/reference/generated/numpy.median.html) as well as the mean. Save your answer in a tuple `q4_2`.

In [8]:
# Code for question 4
np.random.seed(0)

lost = movie_ratings[movie_ratings["movie_title"] == "Lost in Space (1998)"]

ages_lost = lost["age"].dropna().values
n = len(ages_lost)

bootstrap_means = []

for _ in range(10000):
    sample = np.random.choice(ages_lost, size=n, replace=True)
    bootstrap_means.append(np.mean(sample))

q4_1 = (np.percentile(bootstrap_means, 2.5),
        np.percentile(bootstrap_means, 97.5))

ratings_lost = lost["rating"].dropna().values
n = len(ratings_lost)

bootstrap_medians = []

for _ in range(10000):
    sample = np.random.choice(ratings_lost, size=n, replace=True)
    bootstrap_medians.append(np.median(sample))

q4_2 = (np.percentile(bootstrap_medians, 2.5),
        np.percentile(bootstrap_medians, 97.5))


# Leave these lines here for grading and ease of debugging
print("4.1: 95% confidence interval: ", q4_1)
print("4.2: 95% confidence interval: ", q4_2)

4.1: 95% confidence interval:  (np.float64(26.055555555555557), np.float64(36.611111111111114))
4.2: 95% confidence interval:  (np.float64(2.5), np.float64(4.0))


### Question 5 (15 points: 6 autograded, 9 manually graded)

1. (5pts) Consider the null hypothesis that the mean rating of `Muppet Treasure Island (1996)` is the same for `sex='F'` and `sex='M'` users. The alternative hypothesis is that the mean ratings are not equal. Conduct a two-sided t test using [`stats.ttest_ind`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html) to evaluate this using the sample ratings data. Report your p-value in `q5_1` as a `float` or `numpy.float64`. Interpret your p-value at a significance level of $0.05$ in the **Answer 5** cell. **Moreover, explain the underlying assumptions in your t-test in the cell and whether how you ran the function is reasonable given those assumptions including empirical values from the data as needed.** You are welcome to consult the documentation of [`stats.ttest_ind`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html) to help answer this question.

The `Star Wars (1977)` film is quite popular, with a median rating of $5/5$. However, of those that left a rating, male users gave it a slightly higher mean rating of about $4.4$ whereas female users gave the same movie a mean rating of about $4.2$.

2. (4pts) Consider the null hypothesis that $51\%$ of men would rate `Star Wars (1977)` a $5$. The alternative hypothesis is then that the fraction of men who would rate `Star Wars (1977)` a $5$ is not $51\%$. Conduct a two-sided hypothesis test using [`stats.t.cdf`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html) to evaluate this in light of the sample ratings data of male users who rated `Star Wars (1977)`. Report your p-value in `q5_2` as a `float` or `numpy.float64`. Interpret your p-value at a significance level of $0.05$ in the **Answer 5** cell.


3. (4pts) Consider the null hypothesis that women and men were equally likely to rate `Star Wars (1977)` a $5$. The alternative hypothesis is that these proportions are not equal. Conduct a two-sided t test using [`stats.ttest_ind_from_stats`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind_from_stats.html) to evaluate this in light of the sample data of female and male users who rated `Star Wars (1977)`. Report your p-value in `q5_3` as a `float` or `numpy.float64`. Interpret your p-value at a significance level of $0.05$ in the **Answer 5** cell. 


4. (2pts) In the sample ratings data, $51\%$ of women rated `Star Wars (1977)` a $5$. (You are welcome to write code to validate this claim, but you can just accept it as fact.) Therefore, if $51\%$ of men and $51\%$ of women would rate `Star Wars (1977)` a $5$, both null hypotheses in 5.2 and 5.3 would be true. If this is the case, why do you observe a different p-value in the two parts, despite the hypotheses under consideration being ostensibly similar? Briefly explain why you observe this difference in the **Answer 5** cell. 

In [9]:
# Code for question 5
muppet = movie_ratings[movie_ratings["movie_title"] == "Muppet Treasure Island (1996)"]

ratings_F = muppet[muppet["sex"] == "F"]["rating"]
ratings_M = muppet[muppet["sex"] == "M"]["rating"]

t_stat, q5_1 = stats.ttest_ind(ratings_F, ratings_M, equal_var=False)

star_wars = movie_ratings[movie_ratings["movie_title"] == "Star Wars (1977)"]

male_sw = star_wars[star_wars["sex"] == "M"]

n = len(male_sw)
x = np.sum(male_sw["rating"] == 5)

p_hat = x / n
p0 = 0.51

se = np.sqrt(p0 * (1 - p0) / n)

t_stat = (p_hat - p0) / se

q5_2 = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))

female_sw = star_wars[star_wars["sex"] == "F"]

n_M = len(male_sw)
n_F = len(female_sw)

p_M = np.mean(male_sw["rating"] == 5)
p_F = np.mean(female_sw["rating"] == 5)

std_M = np.sqrt(p_M * (1 - p_M))
std_F = np.sqrt(p_F * (1 - p_F))

t_stat, q5_3 = stats.ttest_ind_from_stats(
    mean1=p_M, std1=std_M, nobs1=n_M,
    mean2=p_F, std2=std_F, nobs2=n_F,
    equal_var=False
)


# Leave these lines here for grading and ease of debugging
print("Q5.1: p-value: ", q5_1)
print("Q5.2: p-value: ", q5_2)
print("Q5.3: p-value: ", q5_3)

Q5.1: p-value:  0.07920295536927349
Q5.2: p-value:  0.008010296971218134
Q5.3: p-value:  0.17469334777160545


### Answer 5

5. 

a. The null hypothesis is that the mean rating of `Muppet Treasure Island (1996)` is the same for female and male users. The alternative hypothesis is that the means are different.

Using Welch’s two-sided t-test, we compare the p-value to 0.05. If the p-value is less than 0.05, we reject the null hypothesis and conclude the mean ratings differ by sex. If it is greater than 0.05, we fail to reject the null hypothesis.

*Assumptions:*

1. The male and female samples are independent.

2. Ratings within each group are independent observations.

3. The sampling distribution of the mean is approximately normal. Since the sample sizes are reasonably large, this is justified by the CLT.

4. We used `equal_var=False`, so we do not assume equal population variances. This makes the test appropriate even if the variability differs between males and females.

b. The null hypothesis is that 51\% of men rate `Star Wars (1977)` a 5. The alternative is that the true proportion is not 51\%.

If the p-value is less than 0.05, we reject the null hypothesis and conclude the proportion differs from 51\%. Otherwise, we fail to reject it.

c. The null hypothesis is that men and women are equally likely to rate `Star Wars (1977)` a 5. If the p-value is less than 0.05, we conclude the proportions differ. Otherwise, we do not find sufficient evidence of a difference.

d. The tests in 5.2 and 5.3 are different. 

In 5b, we test one sample proportion against a fixed value (0.51).  
In 5c, we compare two independent samples (men vs. women).  

Because the standard errors are calculated differently in these two settings, the test statistics and the p-values are different, even if both null hypotheses are true.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

# AI Disclosure

Use the **Artificial Intelligence Disclosure (AID) Framework** to explain your use of AI on this assignment. Other headings you can use include:

- *Debugging:* Using AI to help you fix your code so that it works. You should state how you used it for this purpose.

Here are some examples:

*Artificial Intelligence Tools:* ChatGPT v5 via chatgpt.com. *Conceptualization:* I gave chatgpt.com the election data set and asked it for ideas on interesting statistics I could get from the data. *Methodology:* I asked it for help on how to write the code to get the statistic I chose, but I wrote the code myself. *Writing — Review & Editing:* I wrote out my explanation for what the statistic meant, then gave that text and the rubric to chatgpt and asked it to give me feedback on how to update the explanation to conform to the rubric.

*Artificial Intelligence Tools:* ChatGPT v4o via DukeGPT. *Information Collection:* DukeGPT was used to find the function needed to get the index value of the maximum value of a Series and the syntax needed to filter rows in a pandas dataframe using multiple columns. *Debugging:* DukeGPT was used to help me find a bug in my code for Q1 where I copied in the code and error, stated what the code should do, and asked for help.

**Solution**
Artificial Intelligence Tools: ChatGPT v5 via chatgpt.com. Debugging: ChatGPT was used to help me find a bug in my code for Q3 where I copied in the code and error, stated what the code should do, and asked for help. Writing—Review & Editing: I gave my written explanation and a written explanation of my code for Q2, Q3, Q5, as well as the rubric to ChatGPT and asked it to give me feedback on how to update the explanation to conform to the rubric

<!-- END QUESTION -->

## Submitting

You should make sure any code that you write to answer the questions is included in this notebook. You are **required** to go to the Kernel option and choose **"Restart & Run All"**  before submission. Double check that your entire notebook runs correctly and generates the expected output. Finally, make sure to save your work (timestamp at the top tells you the last checkpoint and whether there are unsaved changes). When you finish, submit your assignment at [Gradescope](http://gradescope.com/ "‌"). **Submissions not prepared correctly as above will lose points.**